# Customer Churn Risk Modeling

**Author:** Raya Sergieva  
**Course:** Data Science Final Exam, 2026  
**Date:** May 2026

---

## Project Overview

This notebook develops a customer churn risk model using two independent datasets — a US telecommunications dataset (IBM Telco) and a European retail banking dataset (Churn Modelling). The goal is to predict the probability that a current customer will churn within a defined future window, evaluate that prediction rigorously, and analyze what features drive churn risk.

## Why churn matters

For subscription-based and account-based businesses, retaining an existing customer is typically 5–10 times cheaper than acquiring a new one. If churn risk can be predicted accurately and *calibrated* — meaning the predicted probabilities reflect real-world frequencies — a company can target retention offers cost-effectively. This makes churn modeling one of the most consistently valuable applications of supervised machine learning in industry.

## Approach

1. Load and validate both datasets independently.
2. Perform exploratory data analysis on each.
3. Clean, harmonize feature semantics where possible, and merge into a single analysis-ready dataset.
4. Define the prediction problem mathematically and discuss the choice of loss function and evaluation metrics.
5. Train and compare three model families: logistic regression (interpretable baseline), random forest (non-linear baseline), and gradient boosting (strong non-linear model).
6. Evaluate using ROC-AUC, PR-AUC, Brier score, and calibration plots — accuracy alone is misleading under class imbalance.
7. Interpret the best model using SHAP values to identify churn drivers.
8. Discuss limitations, generalization across domains, and possible extensions.

In [1]:
# Core data science libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings — readable output in the notebook
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")

# Reproducibility — fix the random seed everywhere we can
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Confirm versions for transparency
print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn:    {sns.__version__}")

pandas:     3.0.3
numpy:      2.4.4
matplotlib: 3.10.9
seaborn:    0.13.2


In [2]:
# Paths are relative to the notebook's location (notebooks/)
# so ../data/raw points back up to the project root, then into data/raw
RAW_DATA_DIR = "../data/raw"

# Load both raw datasets
telco = pd.read_csv(f"{RAW_DATA_DIR}/telco_churn.csv")
bank = pd.read_csv(f"{RAW_DATA_DIR}/bank_churn.csv")

# Report shapes so we can verify we got what we expected
print(f"Telco churn dataset:  {telco.shape[0]:>6,} rows × {telco.shape[1]:>2} columns")
print(f"Bank churn dataset:   {bank.shape[0]:>6,} rows × {bank.shape[1]:>2} columns")

Telco churn dataset:   7,043 rows × 21 columns
Bank churn dataset:   10,000 rows × 14 columns


## 1. First look at the raw data

Before any cleaning or modeling, we inspect each dataset independently to understand its structure, column types, missing values, and target distribution. This step catches encoding issues, unexpected dtypes, and obvious data quality problems before they propagate into later analysis.

### 1.1 Telco dataset — preview

In [3]:
# First 5 rows of the Telco dataset
telco.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# Column types and non-null counts
telco.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
# Pandas' view of missing values per column (Telco)
# Sort by missing count descending, show top 10
telco.isna().sum().sort_values(ascending=False).head(10)

customerID         0
gender             0
SeniorCitizen      0
Partner            0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
OnlineSecurity     0
dtype: int64

In [6]:
# Investigate TotalCharges specifically
# It's typed as a string — that suggests something non-numeric is hiding in it
# Try converting to numeric and see how many fail to parse
total_charges_numeric = pd.to_numeric(telco["TotalCharges"], errors="coerce")

n_unparseable = total_charges_numeric.isna().sum()
print(f"Rows where TotalCharges fails to parse as a number: {n_unparseable}")
print()

# Show those problematic rows
print("The rows with the unparseable TotalCharges:")
telco.loc[total_charges_numeric.isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Rows where TotalCharges fails to parse as a number: 11

The rows with the unparseable TotalCharges:


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


### 1.2 Data quality finding: `TotalCharges` column

Investigation reveals that `TotalCharges` is stored as a string rather than numeric. The cause: 11 rows (0.16% of the data) contain whitespace instead of a number. All 11 are customers with `tenure = 0` — i.e. brand-new customers who have not yet been billed — and all 11 are non-churners (which is mechanically consistent: a customer with 0 months of tenure cannot have churned).

This is **structurally absent data**, not missing data in the usual sense. We will impute `TotalCharges = 0` for these rows during cleaning, which is the natural interpretation: a customer who has been billed zero times has a total charge of zero.

In [7]:
# Target distribution (Telco)
print("Churn distribution (Telco):")
print(telco["Churn"].value_counts())
print()
print("Churn rate:")
print(telco["Churn"].value_counts(normalize=True).round(3))

Churn distribution (Telco):
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


## 2. Bank dataset inspection

We now perform the same structural inspection on the bank churn dataset — schema, dtypes, missing values, and target distribution. Doing this independently for each source (before any merging or harmonization) is important: it lets us catch source-specific issues without the confusion of mixed-origin rows.

### 2.1 Bank dataset — preview

In [8]:
# First 5 rows of the Bank dataset
bank.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
# Column types and non-null counts
bank.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


In [10]:
# Missing values per column (Bank)
bank.isna().sum().sort_values(ascending=False).head(10)

RowNumber        0
CustomerId       0
Surname          0
CreditScore      0
Geography        0
Gender           0
Age              0
Tenure           0
Balance          0
NumOfProducts    0
dtype: int64

In [11]:
# Target distribution (Bank)
print("Exited distribution (Bank):")
print(bank["Exited"].value_counts())
print()
print("Exit rate:")
print(bank["Exited"].value_counts(normalize=True).round(3))

Exited distribution (Bank):
Exited
0    7963
1    2037
Name: count, dtype: int64

Exit rate:
Exited
0    0.796
1    0.204
Name: proportion, dtype: float64


### 2.2 Bank dataset — summary of findings

The bank dataset is structurally cleaner than the Telco one:
- No type coercion problems — all numeric columns are stored as numerics.
- No hidden missing values — the `isna()` check returns zero across all columns.
- Three columns are identifiers with no predictive value (`RowNumber`, `CustomerId`, `Surname`) and will be dropped during cleaning.

The target column is `Exited` (1/0) rather than `Churn` (Yes/No). We will harmonize naming during cleaning.

### 2.3 Cross-dataset summary before cleaning

| Aspect              | Telco                          | Bank                                   |
|---------------------|--------------------------------|----------------------------------------|
| Rows                | 7,043                          | 10,000                                 |
| Columns             | 21                             | 14                                     |
| Domain              | US telecommunications          | European retail banking                |
| Target column       | `Churn` (Yes/No)               | `Exited` (1/0)                         |
| Positive class rate | 26.5%                          | 20.4%                                  |
| Data quality issues | `TotalCharges` mis-typed (11 rows); placeholder "No service" values | None observed |

Both datasets exhibit moderate class imbalance, so accuracy alone will be misleading as an evaluation metric. We will use ROC-AUC, PR-AUC, and calibration metrics throughout. The different positive-class rates between domains also raise an interesting question for later: do features and modeling approaches that work well on one domain generalize to the other?